# Capítulo 10 — Internet, bibliotecas e o seu próprio pacote

## Missão
Conversar com a Terra pela web. Você vai subir um **servidor HTTP local** (o "centro de controle"), buscar páginas e JSON com `urllib`, extrair dados de HTML e enviar telemetria com POST. Depois, conhecer a biblioteca padrão e organizar o rover como um **pacote** de verdade.

**No fim deste caderno você terá:** o módulo `meu_rover/rede.py` (cliente do centro de controle) e o pacote do rover com `__init__.py` exportando as peças principais.

**Leitura essencial (opcional, livro em inglês):**
- Seeing How the Web Works · PDF p. 345
- Understanding the Python Standard Library · PDF p. 361
- Importing Python Modules · PDF p. 367
- Making Your Own Modules · PDF p. 370

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap10_internet_bibliotecas_e_o_seu_proprio_pacote.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_rover/
os.makedirs("../meu_rover", exist_ok=True)
open("../meu_rover/__init__.py", "a").close()        # meu_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_rover")

## Ideia em 30 segundos
A web é feita de pedidos e respostas:
- o cliente faz `GET` (buscar) ou `POST` (enviar);
- o servidor responde com um **código** (200 ok, 404 não achado) e um corpo (HTML ou JSON).

Para não depender da internet nem de sites que mudam, você vai subir o seu próprio servidor dentro do caderno, numa *thread*.

O **centro de controle** (já vem pronto): um servidor local na porta 8765 com uma página HTML em `/`, a previsão do tempo em JSON em `/clima` e um `POST /telemetria` que guarda o que receber.

In [ ]:
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

RECEBIDO = []
PAGINA = """<html><head><title>Centro de Controle</title></head><body>
<h1>Missão Aurora</h1><ul class="avisos"><li>Tempestade de poeira às 11h</li><li>Janela de comunicação: 14h</li>
<li>Prioridade: amostras de hematita</li></ul><a href="/clima">clima</a></body></html>"""

class Controle(BaseHTTPRequestHandler):
    def _responder(self, codigo, corpo, tipo):
        dados = corpo.encode("utf-8")
        self.send_response(codigo); self.send_header("Content-Type", tipo)
        self.send_header("Content-Length", str(len(dados))); self.end_headers(); self.wfile.write(dados)

    def do_GET(self):
        if self.path == "/":
            self._responder(200, PAGINA, "text/html; charset=utf-8")
        elif self.path == "/clima":
            self._responder(200, json.dumps({"poeira": 0.42, "vento_kmh": 35, "alerta": True}), "application/json")
        else:
            self._responder(404, "não encontrado", "text/plain; charset=utf-8")

    def do_POST(self):
        corpo = self.rfile.read(int(self.headers["Content-Length"]))
        RECEBIDO.append(json.loads(corpo))
        self._responder(201, json.dumps({"ok": True, "total": len(RECEBIDO)}), "application/json")

    def log_message(self, *args):
        pass

try:
    servidor.shutdown(); servidor.server_close()
except NameError:
    pass
servidor = ThreadingHTTPServer(("127.0.0.1", 8765), Controle)
threading.Thread(target=servidor.serve_forever, daemon=True).start()
BASE = "http://127.0.0.1:8765"
print("centro de controle em", BASE)

### 10.1 **GET de uma página.** Com `urllib.request.urlopen`, busque `BASE + "/"`, mostre o código de status, o `Content-Type` e os 120 primeiros caracteres do HTML.

### 10.2 **Extrair dados do HTML.** Com `html.parser.HTMLParser`, escreva uma classe que guarda o texto de cada `<li>` e o `<title>`. Mostre os avisos do centro de controle.

### 10.3 **JSON pela web e erros HTTP.** Busque `/clima` e converta com `json.loads`. Depois busque `/nada` e trate o `HTTPError` mostrando o código.

### 10.4 **POST.** Envie as 3 primeiras leituras como JSON para `/telemetria` com `urllib.request.Request` (método POST, cabeçalho `Content-Type: application/json`). Mostre a resposta e o que o servidor guardou.

### 10.5 **A biblioteca padrão.** Sem instalar nada, use: `statistics` (mediana e desvio das temperaturas), `random` (com semente, sorteie 3 posições livres do mapa), `collections.Counter` (tipos de amostra) e `pathlib` (liste os arquivos de `../saidas`).

### 10.6 **Cliente do centro de controle.** Escreva `rede.py` com `buscar_json(url, tempo_limite=5)` e `enviar_json(url, dados)`. Os dois devolvem `(ok, dados_ou_erro)`: capture `HTTPError`, `URLError` e `TimeoutError`, para que a falta de conexão **não** derrube o rover.

_A célula grava `meu_rover/rede.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_rover/rede.py


### 10.7 **Transforme a pasta num pacote.** Reescreva o `__init__.py` do pacote para documentar o rover e expor as peças principais (`status`, os sensores, `Interpretador`, `buscar_json`), com `__version__ = "0.5"` e `__all__`.

_A célula grava `meu_rover/__init__.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_rover/__init__.py


### 10.8 **Use o pacote.** Importe o pacote inteiro (`import meu_rover as rover`), mostre a versão e o `__all__`, busque `/clima` e uma porta onde não há servidor (`http://127.0.0.1:9`) com `rover.buscar_json`. Por fim, desligue o servidor (`servidor.shutdown()`).

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
